In [132]:
import pandas as pd
import os

Cleaning the results column for unplayed matches and convert date column into datetime object

In [133]:
# Load the processed all_matches data
all_matches_data = pd.read_csv(os.path.join("..", "data", "processed", "all_matches_raw.csv"))

# Drop rows with missing 'result' values
all_matches_data = all_matches_data.dropna(subset=['result']).reset_index(drop=True)

# Convert 'date' column to datetime format
all_matches_data['date'] = pd.to_datetime(all_matches_data['date'], dayfirst=True)

print(all_matches_data.tail())

          target_team   opponent_team  is_home                date  \
13550       Liverpool           Paris      NaN 2026-04-14 19:00:00   
13551         Arsenal     Sporting CP      NaN 2026-04-15 19:00:00   
13552  Bayern München     Real Madrid      NaN 2026-04-15 19:00:00   
13553           Paris  Bayern München      NaN 2026-04-28 19:00:00   
13554          Atleti         Arsenal      NaN 2026-04-29 19:00:00   

            competition      stage     season  days_since_last_ucl_match  \
13550  Champions_League  QF Game 2  2025-2026                        NaN   
13551  Champions_League  QF Game 2  2025-2026                        NaN   
13552  Champions_League  QF Game 2  2025-2026                        NaN   
13553  Champions_League  SF Game 1  2025-2026                        NaN   
13554  Champions_League  SF Game 1  2025-2026                        NaN   

       ucl_impact_category result  target_team_goals  opponent_team_goals  \
13550                  NaN  0 - 2            

Filling "target_team_goals" and "opponent_team_goals" columns

In [134]:
# Split the 'result' column into 'target_team_goals' and 'opponent_team_goals'
score_parts = all_matches_data['result'].str.split(' - ', expand=True)

# Convert the split parts to integers and assign them to new columns
all_matches_data['target_team_goals'] = score_parts[0].astype(int)
all_matches_data['opponent_team_goals'] = score_parts[1].astype(int)

print(all_matches_data.head())

     target_team opponent_team  is_home                date competition stage  \
0     Fiorentina        Torino      NaN 2020-09-19 18:00:00     Serie_A     1   
1  Hellas Verona          Roma      NaN 2020-09-19 20:45:00     Serie_A     1   
2          Parma        Napoli      NaN 2020-09-20 12:30:00     Serie_A     1   
3          Genoa       Crotone      NaN 2020-09-20 15:00:00     Serie_A     1   
4       Sassuolo      Cagliari      NaN 2020-09-20 18:00:00     Serie_A     1   

      season  days_since_last_ucl_match  ucl_impact_category result  \
0  2020-2021                        NaN                  NaN  1 - 0   
1  2020-2021                        NaN                  NaN  0 - 0   
2  2020-2021                        NaN                  NaN  0 - 2   
3  2020-2021                        NaN                  NaN  4 - 1   
4  2020-2021                        NaN                  NaN  1 - 1   

   target_team_goals  opponent_team_goals  points  ucl_format  \
0                  1 

perspektif

In [135]:
# perspective 1: Home team is the target team
# the original 'target_team' in raw data is the home team
home_perspective_data = all_matches_data.copy()
home_perspective_data['is_home'] = True

# perspective 2: Away team is the target team
# we swap the theams and their respective goals
away_perspective_data = all_matches_data.copy()
away_perspective_data['target_team'] = all_matches_data['opponent_team']
away_perspective_data['opponent_team'] = all_matches_data['target_team']
away_perspective_data['target_team_goals'] = all_matches_data['opponent_team_goals']
away_perspective_data['opponent_team_goals'] = all_matches_data['target_team_goals']
away_perspective_data['is_home'] = False

# Combine the home and away perspectives into one master long-format DataFrame
team_centric_matches = pd.concat([home_perspective_data, away_perspective_data], ignore_index=True)

# sort the data cronologically for each team
# This sorting is CRITICAL for calculating match counts and previous match status later
team_centric_matches = team_centric_matches.sort_values(by=['target_team', 'season', 'date']).reset_index(drop=True)

# check the results for a specific team
print(team_centric_matches[team_centric_matches['target_team'] == 'Galatasaray'].head())

       target_team        opponent_team  is_home                date  \
10384  Galatasaray       Gaziantep F.K.     True 2020-09-12 20:00:00   
10385  Galatasaray  Istanbul Basaksehir    False 2020-09-20 19:00:00   
10386  Galatasaray           Fenerbahçe     True 2020-09-27 19:00:00   
10387  Galatasaray            Kasimpasa    False 2020-10-04 19:00:00   
10388  Galatasaray           Alanyaspor     True 2020-10-19 20:00:00   

      competition stage     season  days_since_last_ucl_match  \
10384   Super_Lig     1  2020-2021                        NaN   
10385   Super_Lig     2  2020-2021                        NaN   
10386   Super_Lig     3  2020-2021                        NaN   
10387   Super_Lig     4  2020-2021                        NaN   
10388   Super_Lig     5  2020-2021                        NaN   

       ucl_impact_category result  target_team_goals  opponent_team_goals  \
10384                  NaN  3 - 1                  3                    1   
10385                 

Calculating match counts. Filling the "target_team_match_count" and "oponent_team_match_count"

In [136]:
# Calculate the cumulative match count for each team
team_centric_matches['target_team_match_count'] = team_centric_matches.groupby(['target_team', 'season']).cumcount() + 1

# Calculate the cumulative match count for the opponent team
# to find it we need a lookup table
# this table maps a team's name and date to their specific match number
match_number_lookup = team_centric_matches[['target_team', 'date', 'target_team_match_count']].copy()

# rename the columns to match the 'opponent' perspective
match_number_lookup.columns = ['opponent_team', 'date', 'opponent_team_match_count']

# merge the lookup table back to the main DataFrame
# we drop the existing 'opponent_team_match_count' column to avoid duplicates
if 'opponent_team_match_count' in team_centric_matches.columns:
    team_centric_matches = team_centric_matches.drop(columns=['opponent_team_match_count'])

team_centric_matches = pd.merge(
    team_centric_matches, 
    match_number_lookup, 
    on=['opponent_team', 'date'], 
    how='left'
)

print(team_centric_matches[team_centric_matches['target_team'] == 'Galatasaray'].tail(6))

       target_team   opponent_team  is_home                date  \
10613  Galatasaray       Liverpool    False 2026-03-18 20:00:00   
10614  Galatasaray     Trabzonspor    False 2026-04-04 20:00:00   
10615  Galatasaray         Göztepe    False 2026-04-08 20:00:00   
10616  Galatasaray     Kocaelispor     True 2026-04-12 20:00:00   
10617  Galatasaray  Gençlerbirligi    False 2026-04-18 20:00:00   
10618  Galatasaray      Fenerbahçe     True 2026-04-26 20:00:00   

            competition       stage     season  days_since_last_ucl_match  \
10613  Champions_League  R16 Game 2  2025-2026                        NaN   
10614         Super_Lig          28  2025-2026                        NaN   
10615         Super_Lig          27  2025-2026                        NaN   
10616         Super_Lig          29  2025-2026                        NaN   
10617         Super_Lig          30  2025-2026                        NaN   
10618         Super_Lig          31  2025-2026                      

Calculate points and determine UCL format

In [137]:
# 1: Calculate points
# Define a function to determine points from the target team's perspective
def determine_match_points(row):
    if row['target_team_goals'] > row['opponent_team_goals']:
        return 3
    elif row['target_team_goals'] == row['opponent_team_goals']:
        return 1
    else:
        return 0

# Apply the function to calculate points for each match 
# I didn't care about the UCL points because they are not relevant and some matches don't even have them (qualifiers, playoffs, QF, SF, F)
team_centric_matches['points'] = team_centric_matches.apply(determine_match_points, axis=1)


# 2: Determine the UCL format
# define the seasons with the new format
new_format_seasons = ['2024-2025', '2025-2026']

# Initialize the column with NA to ensure demostic league matches remain empty
team_centric_matches['ucl_format'] = pd.NA

# create a mask to only target UCL matches
ucl_matches_mask = team_centric_matches['competition'] == 'Champions_League'

# For the matches that are in the UCL, determine if they are in the new or old format based on the season
team_centric_matches.loc[ucl_matches_mask, 'ucl_format'] = team_centric_matches.loc[ucl_matches_mask, 'season'].apply(
    lambda x: 'new' if x in new_format_seasons else 'old'
)

print(team_centric_matches['ucl_format'].value_counts(dropna=False))


ucl_format
<NA>    25360
old      1000
new       750
Name: count, dtype: int64


Calculate how many days have past since the last UCL match 

In [138]:
# 1: create a subset containing only UCL matches
# this will serve as a reference point for previous UCL dates
ucl_only_matches = team_centric_matches[team_centric_matches['competition'] == 'Champions_League'].copy()

# 2: select only the necessary columns for the lookup and rename the date column
ucl_dates_lookup = ucl_only_matches[['target_team', 'date']].copy()
ucl_dates_lookup.columns = ['target_team', 'last_ucl_date']

# 3: sort both the DataFrames by date for merge
team_centric_matches = team_centric_matches.sort_values('date')
ucl_dates_lookup = ucl_dates_lookup.sort_values('last_ucl_date')

# 4: Use merge_asof to find the closest previous UCL match for each row
team_centric_matches = pd.merge_asof(
    team_centric_matches,
    ucl_dates_lookup,
    left_on='date',
    right_on='last_ucl_date',
    by='target_team',
    direction='backward',
    allow_exact_matches=False
)

# 5: Calculate the days since the last UCL match
team_centric_matches['days_since_last_ucl_match'] = (
    team_centric_matches['date'] - team_centric_matches['last_ucl_date']
).dt.days
# 6: categorize the impact based on the days since the last UCL match
def categorize_ucl_impact(days):
    if pd.isna(days) or days > 6:
        return 'no_effect'
    elif days <= 3:
        return 'severe'
    else:  # 4-6 days
        return 'moderate'
    
team_centric_matches['ucl_impact_category'] = team_centric_matches['days_since_last_ucl_match'].apply(categorize_ucl_impact)

# 7: Final verification
team_centric_matches = team_centric_matches.sort_values(['target_team', 'date']).reset_index(drop=True)
print(team_centric_matches[team_centric_matches['target_team'] == 'Galatasaray'].tail(11))

team_centric_matches.to_csv(os.path.join("..", "data", "processed", "team_centric_matches.csv"), index=False)

       target_team        opponent_team  is_home                date  \
10608  Galatasaray             Juventus    False 2026-02-25 20:00:00   
10609  Galatasaray           Alanyaspor     True 2026-02-28 20:00:00   
10610  Galatasaray             Besiktas    False 2026-03-07 20:00:00   
10611  Galatasaray            Liverpool     True 2026-03-10 17:45:00   
10612  Galatasaray  Istanbul Basaksehir     True 2026-03-14 20:00:00   
10613  Galatasaray            Liverpool    False 2026-03-18 20:00:00   
10614  Galatasaray          Trabzonspor    False 2026-04-04 20:00:00   
10615  Galatasaray              Göztepe    False 2026-04-08 20:00:00   
10616  Galatasaray          Kocaelispor     True 2026-04-12 20:00:00   
10617  Galatasaray       Gençlerbirligi    False 2026-04-18 20:00:00   
10618  Galatasaray           Fenerbahçe     True 2026-04-26 20:00:00   

            competition            stage     season  \
10608  Champions_League  Play-off Game 2  2025-2026   
10609         Super_Lig  